# Sleep-by-hour → Prism (worked example)

This notebook turns the pipeline's **sleep export** into GraphPad-Prism-ready
spreadsheets. It is a self-contained example — everything runs top to bottom and
the reshaping helpers live in one cell you can copy into your own analysis code.

**Input:** the `sleep_export/` folder written by the *Finalize → Export Sleep Data*
step. We use `sleep_bins_avg.csv` (per-animal metrics averaged across days, one row
per animal × sleep state × 2-hour clock bin).

**Output:** one `.xlsx` with a sheet per *state × metric* (e.g. `w_frac_time`),
laid out as rows = time bins, columns = animals grouped by experimental group —
the shape you paste straight into Prism.

In [1]:
import pandas as pd
from pathlib import Path
from openpyxl import Workbook

## 1. Point at your export and load the by-hour table

In [2]:
EXPORT_DIR = Path("/media/matt/Data_A/leah/Stxbp1_characterization/sleep_export")

bins = pd.read_csv(EXPORT_DIR / "sleep_bins_avg.csv")
bins.head()

,mouse_id,state,hour_of_day,frac_time,n_bouts,mean_bout_sec,n_days
0,Mice-1248,n,0,0.342824,17.333333,141.572682,3
1,Mice-1248,n,2,0.211250,13.333333,113.324786,3
2,Mice-1248,n,4,0.089676,6.000000,97.208333,3
3,Mice-1248,n,6,0.547222,28.000000,142.282353,3
4,Mice-1248,n,8,0.593472,31.000000,139.986550,3


Columns: `mouse_id`, `state` (`w`/`n`/`r`), `hour_of_day` (the 2-hour bin's start
in clock time: 0, 2, … 22, so `6` is 6 AM lights-on and `0` is midnight), the three
metrics `frac_time`, `n_bouts`, `mean_bout_sec`, and `n_days` (how many days were
averaged for that bin).

The export already rotates each recording so it begins at 6 AM, so every animal is
comparable at the same clock hour. In step 4 the writer can convert this clock hour
to **ZT** (hours since lights-on) via `(hour_of_day - anchor) % 24`, where the anchor
hour becomes ZT 0 — 6 AM by default, configurable, or `None` to keep raw clock
hours.

## 2. Assign each animal to a group

Edit the `groups` dict for your project: keys are the group names (they become the
column blocks in Prism), values are the `mouse_id`s exactly as they appear in the
export. The assertion catches any animal you forgot to place.

In [3]:
groups = {
    "WT": [
        "Mice-1248", "Mice-1253", "Mice-1254", "Mice-1282", "Mice-1284",
        "Mice-1286", "Mice-C266", "Mice-C317", "Mice-C567", "Mice-C576",
        "Mice-C577",
    ],
    "MUT": [
        "Mice-1251", "Mice-1279", "Mice-C259", "Mice-C264", "Mice-C268",
        "Mice-C269", "Mice-C309", "Mice-C315", "Mice-C348", "Mice-C561",
        "Mice-C563", "Mice-C568", "Mice-C569",
    ],
}

animal_to_group = {a: g for g, animals in groups.items() for a in animals}
bins["group"] = bins["mouse_id"].map(animal_to_group)

unassigned = sorted(bins.loc[bins["group"].isna(), "mouse_id"].unique())
assert not unassigned, f"These animals are not in any group: {unassigned}"
print(f"{len(animal_to_group)} animals assigned across groups: {list(groups)}")

24 animals assigned across groups: ['WT', 'MUT']


## 3. Helpers — reshape to the Prism layout

These functions are the only reusable piece. `write_prism_workbook` optionally
converts each clock hour to ZT (hours since lights-on, set by `zt_anchor_hour`; pass
`None` to keep raw clock hours) and writes one sheet per *state × metric*; each sheet
has two header rows (group name, then animal id) above rows of time bins. **To reuse
this in your own analysis, copy just this cell** into a `.py` module.

In [4]:
def bin_label(hour: int) -> str:
    """Turn a 2-hour bin start (e.g. 2) into a Prism row label ('2-4')."""
    return f"{int(hour)}-{int(hour) + 2}"


def hour_ampm(hour: int) -> str:
    """Friendly 12-hour label for a clock hour (6 -> '6am', 0 -> '12am')."""
    period = "am" if hour % 24 < 12 else "pm"
    return f"{hour % 12 or 12}{period}"


def write_prism_workbook(
    df,
    out_path,
    *,
    group_order,
    metrics=("frac_time", "n_bouts", "mean_bout_sec"),
    states=("w", "n", "r"),
    clock_col="hour_of_day",
    zt_anchor_hour=6,
):
    """Write one sheet per (state, metric): rows = time bins, columns = animals
    blocked by group, with group-name and animal-id header rows.

    Time handling depends on ``zt_anchor_hour``:
    - an hour (default ``6``): convert clock hours in ``clock_col`` to ZT (hours
      since lights-on) so that hour becomes ZT 0; column labelled ``Time (0 = 6am)``.
    - ``None``: leave the clock hours as-is; column labelled ``Time (clock hour)``.
    """
    sheet_metric = {"n_bouts": "bouts"}  # nicer sheet names; others left as-is
    df = df.copy()
    if zt_anchor_hour is None:
        time_col = clock_col
        time_label = "Time (clock hour)"
    else:
        df["zt"] = (df[clock_col] - zt_anchor_hour) % 24  # clock hour -> ZT hour
        time_col = "zt"
        time_label = f"Time (0 = {hour_ampm(zt_anchor_hour)})"
    time_bins = sorted(df[time_col].unique())

    wb = Workbook()
    wb.remove(wb.active)  # drop the default empty sheet

    for metric in metrics:
        for state in states:
            sub = df[df["state"] == state]
            # animals per group, in the given group order
            group_animals = {
                g: sorted(sub.loc[sub["group"] == g, "mouse_id"].unique())
                for g in group_order
            }
            # wide table for fast (time bin, animal) lookup
            wide = sub.pivot_table(index=time_col, columns="mouse_id", values=metric)

            name = f"{state}_{sheet_metric.get(metric, metric)}"
            ws = wb.create_sheet(name[:31])
            ws.cell(row=2, column=1, value=time_label)

            col = 2
            for g in group_order:
                for animal in group_animals[g]:
                    ws.cell(row=1, column=col, value=g)
                    ws.cell(row=2, column=col, value=animal)
                    col += 1

            for r, b in enumerate(time_bins, start=3):
                ws.cell(row=r, column=1, value=bin_label(b))
                col = 2
                for g in group_order:
                    for animal in group_animals[g]:
                        val = wide.at[b, animal] if animal in wide.columns else None
                        ws.cell(row=r, column=col, value=None if pd.isna(val) else float(val))
                        col += 1

            ws.freeze_panes = "B3"  # keep bin labels + headers visible while scrolling

    wb.save(out_path)
    return out_path

## 4. Write the workbook

In [5]:
# ZT 0 = lights-on. Default 6 (6 AM); set to whatever clock hour your lights
# turn on (e.g. 7 for 7 AM, 19 for 7 PM). Set to None to skip ZT conversion and
# keep the raw clock hours instead.
ZT_ANCHOR_HOUR = 6

OUT_PATH = EXPORT_DIR / "prism_sleep_by_hour.xlsx"
write_prism_workbook(bins, OUT_PATH, group_order=list(groups), zt_anchor_hour=ZT_ANCHOR_HOUR)
print("Wrote", OUT_PATH)

Wrote /media/matt/Data_A/leah/Stxbp1_characterization/sleep_export/prism_sleep_by_hour.xlsx


Each sheet (`w_frac_time`, `w_bouts`, `r_mean_bout_sec`, …) is ready to
copy-paste into a Prism grouped table: the two header rows give the group and
animal, and each column is one animal's values down the time bins.

To adapt this for the **whole-recording** summary instead, load
`sleep_whole_file.csv` and drop the time axis — one row per animal rather than per
bin.